# Benchmark — Fase 2: trackers (2×2, consistencia)

## ¿Qué decide este notebook?

**Qué tracker conviene** entre `bytetrack` y `botsort`. Como el costo del tracker es
**casi gratis** frente a la detección, no se elige por velocidad sino por
**consistencia** del seguimiento (sin ground-truth): fragmentación y longitud de
tracklet.

## ¿Por qué 2×2 y no solo sobre el detector ganador de la Fase 1?

Porque la consistencia del tracker **depende de qué cajas le da el detector**. Correr
el **2×2 completo** (ambos detectores × ambos trackers) evita la crítica "¿y si el otro
detector daba mejores tracks?" — son solo 2 corridas extra y el resultado queda
inatacable.

## Métricas (sin ground-truth → consistencia, no exactitud)

- **Robustas:** `fps`, `peak_vram_mb` (eficiencia) y `frag_rate`/`tracklet_len`
  (consistencia del tracking).
- **Débiles (suplementarias):** `smoothness`, `mask_iou`, `com_jitter` — confundidas
  por movimiento real, diluidas por clases estáticas y *gameables*. Se reportan con
  reserva.

## Requisitos

Iguales que la Fase 1 (pod/GPU, `.env`, pesos YOLO + SAM3 + los 5 videos). Usa los
**mismos videos** (seed=36) para que ambas fases sean coherentes.

## Paso 1 — Videos y configuraciones (2×2)

In [ ]:
from src.core.batch import run_batch
from src.eval.benchmark import aggregate_config, benchmark_videos, comparison_table

videos = benchmark_videos(n=5, seed=36)  # mismos que la Fase 1
print("videos del benchmark (split=2, seed=36):")
for v in videos:
    print(" ", v)

# 2x2: ambos detectores x ambos trackers. (label, detector, tracker)
CONFIGS = [
    ("sam3_text+bytetrack", "sam3_text", "bytetrack"),
    ("sam3_text+botsort", "sam3_text", "botsort"),
    ("yolo_sam3+bytetrack", "yolo_sam3", "bytetrack"),
    ("yolo_sam3+botsort", "yolo_sam3", "botsort"),
]
print(f"\n{len(CONFIGS)} configuraciones (2 detectores × 2 trackers).")

## Paso 2 — Correr el 2×2 y medir

Por cada config se llama a `run_batch` en `mode="tracking"` con:

- **`sampling="auto"` + `max_frames=MAX_FRAMES`** → prefijo contiguo por *streaming*
  (`iter_frames`): memoria acotada aunque el video sea largo.
- **`include_masks=INCLUDE_MASKS`** → si `True`, añade las métricas de máscara
  (suplementarias, casi gratis); ponlo en `False` para correr más ligero.
- **`run_label=f"trackers/<label>"`** → salidas aisladas por config + skip-done por
  config (reanudable).
- **`progress=True`** → barra de progreso por video (útil para ver el avance del
  tracking en videos largos).

`aggregate_config` produce una fila por config; se persiste al CSV apenas se calcula.

In [ ]:
import gc

import pandas as pd
import torch

from src.utils import PROJECT_ROOT

CSV_PATH = PROJECT_ROOT / "outputs" / "benchmark" / "trackers.csv"
MAX_FRAMES = 2500  # tope contiguo (streaming) para acotar videos largos. None = completo.
INCLUDE_MASKS = True  # métricas de máscara (suplementarias). False = más ligero.
RESUME = False  # True para reanudar tras un crash (salta las configs ya guardadas).

CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
if not RESUME and CSV_PATH.exists():
    CSV_PATH.unlink()
done = set(pd.read_csv(CSV_PATH)["config"]) if CSV_PATH.exists() else set()
if done:
    print("reanudando; ya guardadas:", done)

for label, det, trk in CONFIGS:
    if label in done:
        print(f"skip {label} (ya en el CSV)")
        continue
    print(f"\n===== {label}  (tracking) =====")
    summary = run_batch(
        mode="tracking",
        videos=videos,
        detector=det,
        tracker=trk,
        sampling="auto",  # prefijo contiguo (streaming)
        max_frames=MAX_FRAMES,
        include_masks=INCLUDE_MASKS,
        render_video=False,
        overwrite=not RESUME,
        run_label=f"trackers/{label}",
        progress=True,
    )
    row = aggregate_config(label, summary)
    comparison_table([row]).to_csv(
        CSV_PATH, mode="a", header=not CSV_PATH.exists(), index=False
    )
    print("fila guardada:", row)

    del summary
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Paso 3 — Tabla comparativa

In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)
print(f"tabla de trackers ({len(df)}/{len(CONFIGS)} configs)")
df

## Cómo leer y decidir

| Columna | Qué mide | Mejor cuando | Peso |
|---|---|---|---|
| `fps` | throughput (frames/s) | **mayor** | eficiencia |
| `peak_vram_mb` | VRAM pico | **menor** | eficiencia |
| `tracklet_len` | frames promedio que vive un `obj_id` | **mayor** | **robusta** |
| `frag_rate` | proxy de cambios de ID | **menor** | **robusta** |
| `smoothness` | varianza de la aceleración del centroide | **menor** | débil |
| `mask_iou` | IoU de máscara entre frames consecutivos | **mayor** | débil |
| `com_jitter` | temblor del centro de masa | **menor** | débil |

### Advertencias

- Son métricas **sin GT**: miden **consistencia y eficiencia**, NO exactitud vs.
  humano. Un `frag_rate` bajo no garantiza que el ID sea *correcto*.
- **Decisión del tracker:** mira sobre todo `frag_rate`/`tracklet_len` (la eficiencia
  entre trackers es casi igual). El 2×2 además muestra si el detector elegido en la
  Fase 1 sigue siendo buen sustrato para el tracker.
- La **decisión final es humana**: la tabla informa, no dictamina un ganador
  automático. La exactitud llegará con la evaluación contra ground-truth.